In [1]:
import os
import tempfile
from datetime import date

import boto3
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
POSTGRES_HOST = "postgres"
POSTGRES_PORT = 5432
POSTGRES_DB = "analytics"
POSTGRES_USER = "user"
POSTGRES_PASSWORD = "pass"

MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "username"
MINIO_SECRET_KEY = "password"

BUCKET = "raw"
LOAD_DATE = date.today().isoformat()

In [3]:
pg_url = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

engine = create_engine(pg_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.scalar())

1


In [4]:
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
)

buckets = [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]]
print(buckets)

['marts', 'processed', 'raw']


In [9]:
def upload_table_to_minio(table_schema: str, table_name: str) -> None:
    full_table_name = f"{table_schema}.{table_name}"

    query = f'SELECT * FROM "{table_schema}"."{table_name}"'
    df = pd.read_sql(query, engine)

    df["load_date"] = LOAD_DATE
    
    df.columns = [col.lower() for col in df.columns]

    with tempfile.TemporaryDirectory() as tmpdir:
        parquet_path = os.path.join(tmpdir, f"{table_name}.parquet")
        csv_path = os.path.join(tmpdir, f"{table_name}.csv")

        df.to_parquet(parquet_path, index=False)
        df.to_csv(csv_path, index=False)

        parquet_key = (
            f"postgres/{table_schema}/{table_name}/"
            f"load_date={LOAD_DATE}/data.parquet"
        )

        csv_key = (
            f"postgres/{table_schema}/{table_name}/"
            f"load_date={LOAD_DATE}/data.csv"
        )

        s3.upload_file(parquet_path, BUCKET, parquet_key)
        s3.upload_file(csv_path, BUCKET, csv_key)

    print(f"Uploaded {full_table_name}: {len(df)} rows")

In [10]:
tables_query = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_type = 'BASE TABLE'
ORDER BY table_schema, table_name;
"""

tables = pd.read_sql(tables_query, engine)

for _, row in tables.iterrows():
    upload_table_to_minio(
        table_schema=row["table_schema"],
        table_name=row["table_name"],
    )

Uploaded public.deliveries: 30 rows
Uploaded public.drivers: 5 rows
Uploaded public.oil_stations: 20 rows
Uploaded public.production: 150 rows
Uploaded public.pump_failures: 3 rows
Uploaded public.pump_sensors: 72 rows
Uploaded public.pumps: 5 rows
Uploaded public.vehicles: 5 rows
Uploaded public.well_targets: 90 rows
Uploaded public.well_telemetry: 48 rows
Uploaded public.wells: 5 rows
